# Exploración de F1 · catálogo de pistas (EG-12)

**Objetivo:** entender el CSV de pistas (F1) antes de limpiarlo y anotar las cifras exactas que justifican la decisión del [ADR-0003](../docs/adr/0003-duplicados-f1.md).

**Entrada:** `data/raw/dataset.csv` (no se sube al repositorio; la sección *Catálogo F1* del README explica cómo descargarlo).

**Qué buscamos:** filas y columnas, tipos, nulos, filas repetidas por `track_id`, pistas inválidas, artistas y popularidad. La limpieza en sí está en `src/sonoraplay/catalogo/limpiar_f1.py`; este notebook solo mira.

In [1]:
from pathlib import Path

import pandas as pd

# Funciona tanto si Jupyter se abre en la raíz del repo como en notebooks/
RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
df = pd.read_csv(RAIZ / "data" / "raw" / "dataset.csv")
print(f"{len(df):,} filas y {df.shape[1]} columnas")

114,000 filas y 21 columnas


## 1. Columnas y tipos

Cada fila es una pista **dentro de un género**. Las columnas de audio (`danceability`, `energy`, etc.) vienen de la API de Spotify. pandas adivina bien los tipos: números como `int64`/`float64`, `explicit` como booleano y los textos como `object`.

In [2]:
df.dtypes.to_frame("tipo")

,tipo
Unnamed: 0,int64
track_id,object
artists,object
album_name,object
track_name,object
popularity,int64
duration_ms,int64
explicit,bool
danceability,float64
energy,float64


In [3]:
df.head(3).T

,0,1,2
Unnamed: 0,0,1,2
track_id,5SuOikwiRyPMVoIQDJUgSV,4qPNDBW1i3p13qLCt0Ki3A,1iJBSr7s7jYXzM8EGcbK5b
artists,Gen Hoshino,Ben Woodward,Ingrid Michaelson;ZAYN
album_name,Comedy,Ghost (Acoustic),To Begin Again
track_name,Comedy,Ghost - Acoustic,To Begin Again
popularity,73,55,57
duration_ms,230666,149610,210826
explicit,False,False,False
danceability,0.676,0.42,0.438
energy,0.461,0.166,0.359


## 2. Columna índice sobrante

La primera columna no tiene nombre; pandas la llama `Unnamed: 0`. Es solo el número de fila que quedó al exportar el CSV, así que la limpieza la elimina. Lo comprobamos: coincide exactamente con 0, 1, 2, …

In [4]:
print("¿'Unnamed: 0' es igual al número de fila?", (df["Unnamed: 0"] == range(len(df))).all())

¿'Unnamed: 0' es igual al número de fila? True


## 3. Valores nulos

Solo hay **una** fila con nulos: le faltan `artists`, `album_name` y `track_name`, y además dura 0 ms. No sirve como pista: se descarta.

In [5]:
nulos = df.isna().sum()
nulos[nulos > 0].to_frame("nulos")

,nulos
artists,1
album_name,1
track_name,1


In [6]:
df[df.isna().any(axis=1)].T

,65900
Unnamed: 0,65900
track_id,1kR4gIb7nGxHPI3D2ifs59
artists,NaN
album_name,NaN
track_name,NaN
popularity,0
duration_ms,0
explicit,False
danceability,0.501
energy,0.583


## 4. Duración inválida

Una pista con `duration_ms <= 0` no puede reproducirse 30 segundos (RN-01), así que es inválida. Resulta que es **la misma fila** de los nulos.

In [7]:
invalidas_dur = df[df["duration_ms"] <= 0]
print("Filas con duration_ms <= 0:", len(invalidas_dur), "· índice:", invalidas_dur.index.tolist())

Filas con duration_ms <= 0: 1 · índice: [65900]


## 5. Filas totales frente a `track_id` únicos

Aquí está el problema principal de F1: hay muchas más filas que pistas. La misma pista aparece una vez **por cada género** en el que Spotify la clasificó.

In [8]:
veces = df["track_id"].value_counts()
repetidos = veces[veces > 1]
print(f"Filas totales:                {len(df):>8,}")
print(f"track_id únicos:              {df['track_id'].nunique():>8,}")
print(f"track_id que se repiten:      {len(repetidos):>8,}")
print(f"Filas que ocupan:             {repetidos.sum():>8,}")
print(f"Filas sobrantes (a fusionar): {repetidos.sum() - len(repetidos):>8,}")

Filas totales:                 114,000
track_id únicos:                89,741
track_id que se repiten:        16,641
Filas que ocupan:               40,900
Filas sobrantes (a fusionar):   24,259


### ¿Cuántas veces se repite cada pista?

La mayoría de las repetidas aparece 2 veces; una llega a 9.

In [9]:
repetidos.value_counts().sort_index().rename_axis("veces").to_frame("track_id")

,track_id
veces,
2,11712
3,2984
4,1372
5,431
6,117
7,22
8,2
9,1


### ¿En cuántos géneros distintos?

Ojo: no todas las repeticiones son géneros distintos. Hay filas **idénticas** (mismo `track_id` y mismo género), que son duplicados exactos.

In [10]:
generos = df.groupby("track_id")["track_genre"].nunique()
print(
    "Duplicados exactos (mismo track_id y género):",
    df.duplicated(["track_id", "track_genre"]).sum(),
)
print("Iguales en todas las columnas:", df.drop(columns="Unnamed: 0").duplicated().sum())
generos[generos > 1].value_counts().sort_index().rename_axis("géneros").to_frame("track_id")

Duplicados exactos (mismo track_id y género): 450


Iguales en todas las columnas: 450


,track_id
géneros,
2,11424
3,2955
4,1361
5,431
6,104
7,21
8,2
9,1


## 6. ¿Las filas repetidas difieren en algo más que el género?

Si solo cambiara el género, fusionar sería trivial. Comparamos cada columna dentro de cada `track_id` repetido: **solo `popularity` cambia**, y en pocas pistas. Todo lo demás (artista, álbum, duración, audio) es idéntico.

In [11]:
cols = [c for c in df.columns if c not in ("Unnamed: 0", "track_genre")]
d = df[df["track_id"].isin(repetidos.index)]
distintos = (d.groupby("track_id")[cols].nunique(dropna=False) > 1).sum()
distintos[distintos > 0].to_frame("track_id con valores distintos")

,track_id con valores distintos
popularity,720


In [12]:
pop = d.groupby("track_id")["popularity"].agg(["min", "max"])
print("Diferencia máxima de popularidad dentro de un track_id:", (pop["max"] - pop["min"]).max())

Diferencia máxima de popularidad dentro de un track_id: 44


### Empates de popularidad

Para elegir un **género principal** usamos el género de la fila con mayor popularidad. Como casi siempre la popularidad es igual en todas las filas, casi siempre hay empate: por eso la regla de desempate (primer género en orden alfabético) importa mucho.

In [13]:
maximo = d.groupby("track_id")["popularity"].transform("max")
en_maximo = d[d["popularity"] == maximo].groupby("track_id")["track_genre"].nunique()
print("track_id repetidos:", len(repetidos))
print("con empate en la popularidad máxima entre 2+ géneros:", (en_maximo > 1).sum())

track_id repetidos: 16641
con empate en la popularidad máxima entre 2+ géneros: 15767


## 7. Mismo nombre y artista, distinto `track_id`

Son pistas que *parecen* la misma (por ejemplo, la versión del álbum y la del sencillo, o remasterizaciones). Tienen `track_id` distinto y las reproducciones de F2 llegan con `track_id`, así que **no** las unimos: cada una es una pista aparte.

In [14]:
validas = df.dropna(subset=["track_name", "artists"])
ids = validas.groupby(["track_name", "artists"])["track_id"].nunique()
print("Pares (nombre, artistas) con 2+ track_id:", (ids > 1).sum())
print("track_id involucrados:", ids[ids > 1].sum())
ids[ids > 1].sort_values(ascending=False).head(5).to_frame("track_id distintos")

Pares (nombre, artistas) con 2+ track_id: 4657
track_id involucrados: 13054


,,track_id distintos
track_name,artists,
Rockin' Around The Christmas Tree,Brenda Lee,45
Little Saint Nick - 1991 Remix,The Beach Boys,41
Run Rudolph Run,Chuck Berry,40
Frosty The Snowman,Ella Fitzgerald,34
Let It Snow! Let It Snow! Let It Snow!,Dean Martin,32


## 8. Artistas

`artists` puede traer varios artistas separados por `;`. El **artista principal** es el texto antes del primer `;`: a él se le asigna un titular de derechos. También hay unos pocos valores con espacios al final (`"Terra "`), que la limpieza recorta.

In [15]:
artistas = df["artists"].dropna()
principal = artistas.str.split(";").str[0].str.strip()
print("Cadenas distintas en 'artists':", artistas.nunique())
print("Artistas principales únicos:", principal.nunique())
print("Valores con espacios al inicio o al final:", (artistas != artistas.str.strip()).sum())

Cadenas distintas en 'artists': 31437
Artistas principales únicos: 17648
Valores con espacios al inicio o al final: 8


In [16]:
principal.value_counts().head(10).to_frame("filas")

,filas
artists,
Wolfgang Amadeus Mozart,354
J Balvin,347
George Jones,343
Daddy Yankee,314
Pritam,302
The Beatles,280
Feid,280
Ella Fitzgerald,258
ILLENIUM,258


## 9. Distribución de popularidad

Va de 0 a 100. Muchas filas tienen popularidad 0 (pistas poco escuchadas). La popularidad solo se usa para elegir el género principal; no afecta las regalías.

In [17]:
df["popularity"].describe().to_frame("popularity")

,popularity
count,114000.000000
mean,33.238535
std,22.305078
min,0.000000
25%,17.000000
50%,35.000000
75%,50.000000
max,100.000000


In [18]:
print("Filas con popularidad 0:", (df["popularity"] == 0).sum())
tramos = pd.cut(df["popularity"], bins=list(range(0, 101, 10)), include_lowest=True)
tramos.value_counts(sort=False).to_frame("filas")

Filas con popularidad 0: 16020


,filas
popularity,
"(-0.001, 10.0]",23462
"(10.0, 20.0]",10715
"(20.0, 30.0]",17579
"(30.0, 40.0]",15570
"(40.0, 50.0]",18904
"(50.0, 60.0]",14200
"(60.0, 70.0]",8724
"(70.0, 80.0]",3892
"(80.0, 90.0]",886


## 10. Resumen para el ADR-0003

| Cifra | Valor |
|---|---|
| Filas leídas | 114.000 |
| Filas inválidas (sin artista ni nombre, y duración 0) | 1 |
| `track_id` únicos | 89.741 (89.740 sin la fila inválida) |
| `track_id` repetidos | 16.641 |
| Filas sobrantes a fusionar | 24.259 (450 son duplicados exactos) |
| Columnas que difieren entre repetidos | solo `popularity` (720 pistas) |
| Repetidos con empate en la popularidad máxima | 15.767 |
| Pares (nombre, artistas) con 2+ `track_id` | 4.657 (13.054 pistas) |
| Artistas principales únicos | 17.648 |
| Géneros | 114 |

**Conclusión:** una fila por `track_id` con un género principal y una tabla puente con todos los géneros (opción B del ADR-0003).